<a href="https://colab.research.google.com/github/fre-denni/genAIforUX-research/blob/main/logbook_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install libraries

In [ ]:
!pip install -q pymupdf pdf2image transformers==4.44.2 google-genai spacy pandas scikit-learn pillow openpyxl
!python -m spacy download en_core_web_sm
!pip install timm einops

We are using an older version of transformers from hugging face to be sure that it's compatible with the Florence2 model from microsoft

### Import necessary libraries

In [ ]:
import os
import fitz  # PyMuPDF
import pandas as pd
from google import genai
from google.genai import types
import spacy
from transformers import AutoProcessor, AutoModelForCausalLM
import transformers.dynamic_module_utils as dynamic_utils
from PIL import Image
import torch
import json
import re
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from IPython.display import clear_output
clear_output()
print("Environment setup correctly!")

Setup folder



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#folder variables, change as you wish
main_folder = "logbook analysis"

# Define the base folder
base_folder = f"/content/drive/MyDrive/{main_folder}/"

# Define input and output folders
input_folder = os.path.join(base_folder, "logbook")
output_folder = os.path.join(base_folder, "output")
output_images_folder = os.path.join(base_folder, "output_images")

# Create output folders if they don't exist
os.makedirs(output_folder, exist_ok=True)
os.makedirs(output_images_folder, exist_ok=True)

print(f"Base folder: {base_folder}")
print(f"Input folder set to: {input_folder}")
print(f"Output folder created/verified: {output_folder}")
print(f"Output images folder created/verified: {output_images_folder}")


Add your API

In [ ]:

from google.colab import userdata
from google import genai

NOME_SECRET = 'GEMINI_API'

try:
    # Recupera la chiave
    api_key = userdata.get(NOME_SECRET)

    if not api_key:
        print(f"❌ ERRORE: La chiave non è stata trovata. Controlla che il nome sia esattamente '{NOME_SECRET}' e che la levetta 'Notebook access' sia attiva.")
    else:
        # Stampiamo inizio e fine per scovare eventuali spazi vuoti o virgolette
        print(f"🔍 Controllo formato: La chiave inizia con '{api_key[:5]}...' e finisce con '...{api_key[-5:]}'")

        # Inizializza il client
        client = genai.Client(api_key=api_key)
        print("✅ Client inizializzato in locale.")

        # Facciamo una vera chiamata di test per validare la chiave sul server
        print("⏳ Provo a contattare Google AI Studio...")
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents='Rispondi solo con "API funzionante al 100%!"'
        )
        print(f"🤖 Risposta da Gemini: {response.text}")

except userdata.SecretNotFoundError:
    print(f"❌ ERRORE CRITICO: Il secret chiamato '{NOME_SECRET}' non esiste nei tuoi Segreti di Colab.")
except Exception as e:
    print(f"❌ ERRORE API (Chiave invalida o problemi di rete): {e}")

Upload and define function of Florence-2

Configure spacy

In [ ]:
nlp = spacy.load("en_core_web_sm") # English language
ruler = nlp.add_pipe("entity_ruler", before="ner")

# Add custom rules for design methodologies
patterns = [
    # A1 - User Journey Maps & varianti
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": "journey"}, {"LOWER": {"IN": ["map", "maps"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "ujm"}]},

    # A1 -Storyboards
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["storyboard", "storyboards"]}}]},

    # A2 - Mental Model Diagrams (cattura sia "mental model" che "mental model diagram")
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mental"}, {"LOWER": "model"}, {"LOWER": {"IN": ["diagram", "diagrams"]}, "OP": "?"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mmd"}]},

    # A2 -Mind & Empathy Maps
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mind"}, {"LOWER": {"IN": ["map", "maps"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "empathy"}, {"LOWER": {"IN": ["map", "maps"]}}]},

    # A1 - User Jobs & JTBD
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": {"IN": ["job", "jobs"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "jobs"}, {"LOWER": "to"}, {"LOWER": "be"}, {"LOWER": "done"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "jtbd"}]},

    # A1 - User Stories
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": {"IN": ["story", "stories"]}}]},

    # A1 - Task Analysis (cattura sia "task analysis" che "task analysis diagram")
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "task"}, {"LOWER": "analysis"}, {"LOWER": {"IN": ["diagram", "diagrams"]}, "OP": "?"}]},

    # A2 - Concept / Conceptual Maps
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["concept", "conceptual"]}}, {"LOWER": {"IN": ["map", "maps"]}}]},

    # A3 - Research
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "literature"}, {"LOWER": "review"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "benchmarking"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["persona", "personas"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "digital"}, {"LOWER": "ethnography"}]},

    # A4 - Mapping (System, Service, Stakeholder) - cattura map, maps e mapping
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "system"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "service"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "stakeholder"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},

    # A2 - Affinity Diagram
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "affinity"}, {"LOWER": {"IN": ["diagram", "diagrams"]}}]}
]
ruler.add_patterns(patterns)

ruler.add_patterns(patterns)
print("Added custom config to spacy")

### Global function and prompts

Set up and force the JSON analysis for Gemini, and the various prompts for the tasks.

In [ ]:
# Force the config of output as json
json_config = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.0 # more deterministic outputs
)

In [ ]:
# define the element present in the logbooks
def classify_assignment(page_text):
  prompt = f"""You are an expert analyst of student logbooks from a UX Design course.
    Your task is to analyze the OCR text extracted from a single page and determine which Assignment bucket it belongs to.

    Use the following ALLOWED CATEGORIES. I have listed the specific UX methodologies that belong to each category.
    Choose ONLY ONE category based on the presence of these keywords or explicit assignment titles:

    - 'A1': Temporal/Chronological maps.
      Keywords: Assignment 1, A1, Assignment 01,  Chrono Maps, User Journey Map, UJM, Storyboard, User Jobs, Jobs to be done, JTBD, User Stories, Task Analysis.

    - 'A2': Non-temporal/Conceptual maps.
      Keywords: Assignment 2, A2, Assignment 02, Mental Model Diagram, MMD, Mind Map, Empathy Map, Concept Map, Conceptual Map, Affinity Diagram.

    - 'A3': Research and Divergent mapping.
      Keywords: Assignment 3, A3, Assignment 03, Literature Review, Benchmarking, Personas, Digital Ethnography, Client Research.

    - 'A4': System and Service Mapping.
      Keywords: Assignment 4, A4, Assignment 04, System Map, System Mapping, Service Map, Service Blueprint, Stakeholder Map, App Mockup.

    - 'AI': Use of AI and Critical Reflections.
      Keywords: The use of AI, Reflections, Prompting, AI usage.
      *CRITICAL RULE: This category has TOP PRIORITY. If the main focus of the text is discussing the interaction with AI tools, prompting strategies, or reflecting on AI outputs, you MUST classify it as 'AI', even if it mentions A1, A2, A3, or A4.*

    - 'Intro': Structural pages.
      Keywords: Index, Table of Contents, Cover Page, Acknowledgments, Logbook Introduction.

    - 'Unknown': Use only if the text is illegible, missing, or completely out of context.

    PAGE TEXT TO ANALYZE:
    ---
    {page_text}
    ---

    Return ONLY a valid JSON object with this exact structure:
    {{
        "assignment": "CategoryName",
        "reasoning": "Brief reason in 10 words or fewer",
        "confidence": "High, Medium, or Low"
    }}
    """

  try:
      response = client.models.generate_content(
          model='gemini-2.5-flash',
          contents=prompt,
          config=json_config
      )
      return json.loads(response.text)
  except Exception as e:
      print(f"Gemini API Error: {e}")
      return {
          "assignment": "Unknown",
          "reasoning": "API Error",
          "confidence": "Low"
      }

---

# And now the OCR loop!



In [ ]:


#GEMINI PROMPT
vlm_prompt = """You are an expert UX design document parser and Smart OCR.
Look at the provided image of a student's logbook page. You must follow these tasks as they are LAWS.

Perform 2 tasks sequentially:

1. READ NARRATIVE TEXT (LAYOUT AWARENESS): Extract the student's main narrative text, headings, and paragraphs.
   CRITICAL RULE: DO NOT extract text that is visually embedded inside screenshots, UI mockups, data charts, tables, or diagrams here. Only extract the student's actual report text and commentary.
   If there is no readable narrative text, return an empty string "".

2. LOCATE IMAGES AND EXTRACT EMBEDDED TEXT: If there are significant diagrams, maps, tables, or mockups on the page, provide their bounding box coordinates. Do NOT box small icons, decorative elements, or single photos without UX context.
   For each image found, you must provide:
   - 'type': a brief categorization (e.g., "Journey Map", "App Mockup", "Data Table").
   - 'box_1000': the bounding box coordinates. Format MUST be [ymin, xmin, ymax, xmax] with values from 0 to 1000 representing proportions of the image dimensions.
   - 'embedded_text': ALL readable text strictly contained INSIDE that specific diagram, table, or mockup. If there is no text inside the image, return an empty string "".

Return ONLY a JSON with this exact structure:
{
  "text": "the extracted narrative text from the page (ignoring images)",
  "img": [
    {
      "type": "Journey Map",
      "box_1000": [100, 50, 450, 950],
      "embedded_text": "all text read from inside this specific journey map..."
    }
  ]
}
"""

# Analyse everything directly with VLM
def process_page_vlm(image_path, prompt):
  """Pass the image from the entire page to Gemini"""
  img = Image.open(image_path)

  try:
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[prompt,img],
        config=json_config
    )
    return json.loads(response.text)
  except Exception as e:
    print(f"Gemini API Error: {e}")
    return None

def extract_pdf_vlm_pipeline(pdf_path, output_dir="output_images"):
  student_id = os.path.basename(pdf_path).replace(".pdf","")
  doc = fitz.open(pdf_path)

  master_records = []

  print(f"Processing {student_id} ({len(doc)} pages)...")

  for page_num in range(len(doc)):
      print(f" -> VLM analysis of page {page_num + 1}...")

      # High quality rendering of the page
      page = doc.load_page(page_num)
      pix = page.get_pixmap(dpi=250)
      img_page = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

      #Save the page
      temp_page_path = f"/tmp/temp_page_{student_id}_{page_num}.png"
      img_page.save(temp_page_path)

      #Run VLM
      page_data = process_page_vlm(temp_page_path, vlm_prompt)

      if not page_data:
        continue

      extracted_text = page_data.get("text", "").strip()
      images = page_data.get("img", [])

      assignment_label = "idk"
      reasoning = ""

      if extracted_text:
        gemini_classification = classify_assignment(extracted_text)
        assignment_label = gemini_classification.get("assignment", "idk")
        reasoning = gemini_classification.get("reasoning", "")

      # Save the temporary record
      master_records.append({
          "ID": student_id,
          "Page": page_num + 1,
          "Type": "Paragraph",
          "Text": extracted_text,
          "Path_Image": temp_page_path,
          "Assignment": assignment_label,
          "Reasoning": reasoning,
      })

      #Cut dinamically the images
      img_w, img_h = img_page.size
      PADDING = 20

      for idx, imm in enumerate(images):
        box = imm.get("box_1000")
        if box and len(box) == 4:
          ymin, xmin, ymax, xmax = box
          #Calculate coordinates
          base_left = (xmin / 1000.0) * img_w
          base_top = (ymin / 1000.0) * img_h
          base_right = (xmax / 1000.0) * img_w
          base_bottom = (ymax / 1000.0) * img_h
          #Add padding
          left = max(0, base_left - PADDING)
          top = max(0, base_top - PADDING)
          right = min(img_w, base_right + PADDING)
          bottom = min(img_h, base_bottom + PADDING)

          try:
            crop_img = img_page.crop((left, top, right, bottom))
            #clean file name
            entry_type = imm.get("type", "schema")
            type_clean = entry_type.replace(" ", "_").replace("-", "_").lower()
            type_clean = re.sub(r'[^a-z0-9_]', '',type_clean)
            type_clean = type_clean[:30]
            #if the string turns out empty
            if not type_clean:
              type_clean = "schema"
            #create filename
            filename = f"{student_id}_pag{page_num+1}_{type_clean}_{idx}.png"
            filepath = os.path.join(output_dir, filename)
            crop_img.save(filepath)

            embeded_text = imm.get("embedded_text", "").strip()

            #Save record
            master_records.append({
                "ID": student_id,
                "Page": page_num + 1,
                "Layout": "Image_Scheme",
                "OCR_Text": f"[{entry_type}] {embeded_text}",
                "Path_Image": filepath,
                "Assignment": assignment_label,
                "Reasoning": ""
            })
          except Exception as e:
            print(f"Errore ritaglio immagine a pag {page_num+1}: {e}")

  print(f"Complete PDF {student_id}")

  #create master dataframe
  df_master = pd.DataFrame(master_records)
  return df_master

Quick prova

In [ ]:
import os

proof = os.path.join(input_folder, "11167713.pdf") #quick and dirty 3mb logbook

if os.path.exists(proof):
  # Chiamiamo la funzione corretta per l'elaborazione dei PDF
  indexed = extract_pdf_vlm_pipeline(proof, output_dir=output_images_folder)

  # Ensure the 'results' directory exists before saving the CSV
  os.makedirs("results", exist_ok=True)

  #Save the result for security
  indexed.to_csv("results/master_index.csv", index=False)

  #Show preview
  display(indexed.head(100))
else:
  print(f"File not founded: {proof}")